## Spectrogram and MFCCs of `Rice_university.m4a`

**Task**
1. Calculate and display the spectrogram of the recording.
2. Determine the first **12 MFCCs** for every frame of length **25 ms** with a stride (hop) of **10 ms**.
3. Display the MFCCs as a colorful 2D image, using **every 3rd frame**.

We use `librosa`, as in `lec09_speech_analysis`.

In [ ]:
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import IPython.display as ipd

AUDIO_FILE = 'Rice_university.m4a'

### Loading the recording

`.m4a` files are AAC-compressed. Recent versions of `librosa` read audio through `soundfile`, which cannot decode AAC, so we first convert the file to WAV with `ffmpeg` (installed on Colab; on a Mac use `brew install ffmpeg`).

We also **resample to 16 kHz**, the standard rate for speech (as in the lecture, where recordings were resampled to reduce the data volume). At 16 kHz a 25 ms frame is exactly 400 samples and a 10 ms stride is exactly 160 samples. At 44.1 kHz, 25 ms would be 1102.5 samples, which is not a whole number.

In [ ]:
SR = 16000
WAV_FILE = 'Rice_university_16k.wav'

# convert m4a -> mono 16 kHz wav
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', AUDIO_FILE,
                '-ac', '1', '-ar', str(SR), WAV_FILE], check=True)

y, sr = librosa.load(WAV_FILE, sr=SR)
print('sample rate :', sr, 'Hz')
print('samples     :', len(y))
print('duration    : %.2f s' % (len(y) / sr))
ipd.Audio(y, rate=sr)

In [ ]:
plt.figure(figsize=(12, 3))
librosa.display.waveshow(y, sr=sr)
plt.title('Waveform')
plt.xlabel('Time (s)')
plt.show()

### Frame parameters

| | duration | samples at 16 kHz |
|---|---|---|
| frame (window) length | 25 ms | `win_length = 400` |
| stride (hop) | 10 ms | `hop_length = 160` |
| FFT size | | `n_fft = 512` (next power of 2 ≥ 400; the 400-sample frame is zero-padded) |

Consecutive frames overlap by 15 ms. We use `center=False`, so frame $i$ covers exactly $[10i,\ 10i + 25]$ ms of the recording.

In [ ]:
FRAME_MS, STRIDE_MS = 25, 10
win_length = int(FRAME_MS / 1000 * sr)    # 400
hop_length = int(STRIDE_MS / 1000 * sr)   # 160
n_fft = 512
print('win_length =', win_length, '  hop_length =', hop_length, '  n_fft =', n_fft)

### 1. Spectrogram

The spectrogram is the magnitude of the short-time Fourier transform (STFT): an FFT of every 25 ms frame (Hann window), stacked side by side. We convert it to decibels so quiet and loud components are both visible.

In [ ]:
S = np.abs(librosa.stft(y, n_fft=n_fft, win_length=win_length,
                        hop_length=hop_length, window='hann', center=False))
S_db = librosa.amplitude_to_db(S, ref=np.max)
print('spectrogram shape (frequency bins, frames):', S_db.shape)

plt.figure(figsize=(12, 5))
librosa.display.specshow(S_db, sr=sr, hop_length=hop_length,
                         x_axis='time', y_axis='hz', cmap='magma')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram (25 ms frames, 10 ms stride)')
plt.show()

### 2. First 12 MFCCs for every frame

`librosa.feature.mfcc` follows the steps from the lecture: STFT of each frame → power spectrum → mel filter bank → log → discrete cosine transform. `n_mfcc=12` keeps the first 12 coefficients.

In [ ]:
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=12, n_fft=n_fft,
                            win_length=win_length, hop_length=hop_length,
                            window='hann', center=False)
print('MFCC shape (coefficients, frames):', mfcc.shape)

### 3. Every 3rd frame, as a colorful 2D image

`mfcc[:, ::3]` keeps frames 0, 3, 6, … The time between kept frames is therefore $3 \times 10 = 30$ ms, so we pass `hop_length=3*hop_length` to get a correct time axis.

In [ ]:
mfcc_3 = mfcc[:, ::3]
print('every 3rd frame, shape (coefficients, frames):', mfcc_3.shape)

plt.figure(figsize=(12, 4))
librosa.display.specshow(mfcc_3, sr=sr, hop_length=3 * hop_length,
                         x_axis='time', cmap='viridis')
plt.yticks(np.arange(12) + 0.5, np.arange(1, 13))
plt.ylabel('MFCC coefficient')
plt.colorbar()
plt.title('First 12 MFCCs, every 3rd frame (25 ms frames, 10 ms stride)')
plt.show()

The first coefficient (MFCC 1, the overall log-energy of the frame) is much larger than the others, so it dominates the color scale. To make the structure in the other coefficients visible, we also show the same matrix with each coefficient (row) **standardized** to zero mean and unit variance. This is the usual "feature normalization with mean and std" step from the lecture.

In [ ]:
mfcc_3_norm = (mfcc_3 - mfcc_3.mean(axis=1, keepdims=True)) / mfcc_3.std(axis=1, keepdims=True)

plt.figure(figsize=(12, 4))
librosa.display.specshow(mfcc_3_norm, sr=sr, hop_length=3 * hop_length,
                         x_axis='time', cmap='coolwarm')
plt.yticks(np.arange(12) + 0.5, np.arange(1, 13))
plt.ylabel('MFCC coefficient')
plt.colorbar()
plt.title('First 12 MFCCs, every 3rd frame, each coefficient standardized')
plt.show()

### Comparing spectrogram and MFCCs

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
img0 = librosa.display.specshow(S_db, sr=sr, hop_length=hop_length,
                                x_axis='time', y_axis='hz', cmap='magma', ax=axes[0])
axes[0].set_title('Spectrogram'); fig.colorbar(img0, ax=axes[0], format='%+2.0f dB')
img1 = librosa.display.specshow(mfcc_3_norm, sr=sr, hop_length=3 * hop_length,
                                x_axis='time', cmap='coolwarm', ax=axes[1])
axes[1].set_yticks(np.arange(12) + 0.5); axes[1].set_yticklabels(np.arange(1, 13))
axes[1].set_ylabel('MFCC'); axes[1].set_title('12 MFCCs, every 3rd frame (standardized)')
fig.colorbar(img1, ax=axes[1])
plt.tight_layout(); plt.show()

### Observations
* In the **spectrogram**, speech appears as bursts of energy. Voiced sounds (vowels such as the "i" in *Rice* or the "u" in *university*) show horizontal bands: the harmonics of the speaker's pitch, with stronger regions (formants) below about 4 kHz. The **"s"** sounds (*Rice*, *university*) show up as noise-like energy at high frequencies (above about 4 kHz) with no harmonic stripes. Pauses are dark.
* The **MFCC image** compresses each 257-bin spectrum column into 12 numbers. MFCC 1 follows the loudness of the frame (high during speech, low in silences). The higher coefficients describe the *shape* of the spectral envelope (where the formants are). That is why their pattern changes when one sound changes to the next, while the speaker's pitch harmonics are smoothed away.
* Using every 3rd frame gives a 30 ms spacing between columns. The image is coarser than the full-resolution MFCC matrix, but the sequence of sounds is still clearly visible, and the data is 3 times smaller.